# SCOPE Quick Start Guide

This notebook demonstrates how to use SCOPE interactively.

## Setup

In [10]:
import anndata as ad
import numpy as np
import pandas as pd
from main import SCOPE
import scanpy as sc

## Load Your Data

Load your AnnData object. It must have:
- `obs['terminal_state_cluster']`: Cell labels ('TBD' for unknown)
- `obs['palantir_pseudotime']`: Pseudotime from Palantir
- `obsm[latent_key]`: Latent space (e.g., 'vae_latent_space', 'X_pca')
- Features: Either `X` or `obsm[feature_key]`

In [11]:
# Load your data
data_path = '/Users/zhaoyimin/Desktop/data_palantir_pseudotime_leiden_clustering_mellon_vae.h5ad'  # Change this
data = sc.read_h5ad(data_path)

print(f"Loaded: {data.n_obs} cells × {data.n_vars} features")
print(f"\nAvailable obs columns: {list(data.obs.columns)}")
print(f"Available obsm keys: {list(data.obsm.keys())}")

Loaded: 4142 cells × 16106 features

Available obs columns: ['n_counts', 'leiden', 'cluster', 'palantir_pseudotime', 'palantir_entropy', 'mellon_log_density', 'mellon_log_density_clipped']
Available obsm keys: ['DM_EigenVectors', 'DM_EigenVectors_multiscaled', 'X_pca', 'X_umap', 'imputed_hvg', 'palantir_fate_probabilities', 'vae_latent_space']


In [12]:
data.obs['cluster']

Run4_120703408880541     TBD
Run4_120703409056541     TBD
Run4_120703409580963    Mono
Run4_120703423990708    Mono
Run4_120703436876077     TBD
                        ... 
Run5_241098904976174     TBD
Run5_241106375007076     Ery
Run5_241114577000174     TBD
Run5_241114577004764     TBD
Run5_241114589051630     Ery
Name: cluster, Length: 4142, dtype: category
Categories (4, object): ['DC', 'Ery', 'Mono', 'TBD']

## Prepare Terminal State Labels

Create the `terminal_state_cluster` column if needed:

In [13]:
data.obs['terminal_state_cluster'] = data.obs['cluster'].astype(str)
print("Terminal state distribution:")
print(data.obs['terminal_state_cluster'].value_counts())

Terminal state distribution:
terminal_state_cluster
TBD     3295
Mono     341
DC       266
Ery      240
Name: count, dtype: int64


## Initialize SCOPE

Choose the appropriate parameters based on your data:

In [14]:
# Case 1: Using imputed features in obsm
scope = SCOPE(
    data=data,
    feature_key='imputed_hvg',      # Your feature matrix in obsm
    latent_key='vae_latent_space',  # Your latent space in obsm
    alpha=0.1,                       # Label spreading parameter
    iter_graph=100,                  # Label spreading iterations
    initial_trees=100,               # Initial RF trees
    trees_per_iteration=10,          # Trees added per iteration
    n_neighbors=10,                  # k-NN neighbors
    use_X=False                      # Use obsm[feature_key]
)

# Case 2: Using data.X directly (no imputation)
# scope = SCOPE(
#     data=data,
#     latent_key='X_pca',
#     alpha=0.1,
#     use_X=True  # Use data.X directly
# )

print("✓ SCOPE initialized")

✓ SCOPE initialized


## Run SCOPE Pipeline

### Prepare data

In [15]:
scope.prepare_data()

Mapping between terminal_state_cluster and numbers: {'DC': 0, 'Ery': 1, 'Mono': 2}
Prepared data: 847 labeled cells, 3295 unlabeled cells


### Initiate conformal result

In [16]:
scope.initialize_conformal_result()

Initialized conformal results with recruitment size: 64


### Build graph

In [17]:
scope.build_graph()

Building graph with 10 neighbors...
Graph construction complete


### Initialize Classifiers

In [18]:
scope.initialize_classifiers()

Initialized 3 classifiers with 100 trees each


### Run SCOPE

In [ ]:
while scope.run_scope():
    print(f"Iteration {scope.iteration} complete")

## Save Results

In [ ]:
# Save to disk
output_dir = './scope_output'
scope.save_results(output_dir)

print(f"Results saved to {output_dir}")
print("  - data_complete_results.h5ad")
print("  - conformal_result.pkl")

## Check fata bias

In [21]:
data.obsm['dummy_label']

,DC,Ery,Mono
Run4_120703408880541,0.000000,1.000000,0.000000
Run4_120703409056541,0.031621,0.398293,0.570086
Run4_120703409580963,0.000000,0.000000,1.000000
Run4_120703423990708,0.000000,0.000000,1.000000
Run4_120703436876077,0.047988,0.458471,0.493542
...,...,...,...
Run5_241098904976174,0.119658,0.225635,0.654708
Run5_241106375007076,0.000000,1.000000,0.000000
Run5_241114577000174,0.375269,0.434789,0.189943
Run5_241114577004764,0.288663,0.000000,0.711337
